# Aligning two Visium datasets (squidpy + moscot)

In this notebook, we align two spot resolution spatial transcriptomics datasets of serial sections of breast cancer.

The original notebook used STalign affine-only. Here we use `squidpy` with `moscot` optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load data

In [ ]:
# Source
fname = '../visium_data/slice1_coor.csv'
df1 = pd.read_csv(fname)

xI = np.array(df1[df1.columns[0]])
yI = np.array(df1[df1.columns[1]])

coords_source = np.column_stack([xI, yI])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)))
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(xI, yI, s=20, alpha=0.2, label='source')
ax.legend(markerscale=1)
ax.set_aspect('equal')

In [ ]:
# Target
fname = '../visium_data/slice2_coor.csv'
df2 = pd.read_csv(fname)

xJ = np.array(df2[df2.columns[0]])
yJ = np.array(df2[df2.columns[1]])

coords_target = np.column_stack([xJ, yJ])
adata_target = ad.AnnData(X=np.zeros((len(coords_target), 1)))
adata_target.obsm['spatial'] = coords_target

fig, ax = plt.subplots()
ax.scatter(xJ, yJ, s=20, alpha=0.2, c='#ff7f0e', label='target')
ax.legend(markerscale=1)
ax.set_aspect('equal')

In [ ]:
# Overlay
fig, ax = plt.subplots()
ax.scatter(xI, yI, s=20, alpha=0.2, label='source')
ax.scatter(xJ, yJ, s=20, alpha=0.1, label='target')
ax.legend(markerscale=1)
ax.set_aspect('equal')

## Align using moscot

In [ ]:
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(xI, yI, s=20, alpha=0.1, label='source')
ax.scatter(aligned[:, 0], aligned[:, 1], s=20, alpha=0.1, label='source aligned')
ax.scatter(xJ, yJ, s=20, alpha=0.1, label='target')
lgnd = plt.legend(scatterpoints=1, fontsize=10)
for handle in lgnd.legend_handles:
    handle.set_sizes([20.0])
ax.set_aspect('equal')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 8))
ax[0].scatter(xI, yI, s=20, alpha=0.1, label='source')
ax[0].scatter(xJ, yJ, s=20, alpha=0.1, label='target')
ax[0].set_title('Before alignment')
ax[0].set_aspect('equal')
ax[0].legend(markerscale=1)

ax[1].scatter(aligned[:, 0], aligned[:, 1], s=20, alpha=0.1, label='source aligned')
ax[1].scatter(xJ, yJ, s=20, alpha=0.1, label='target')
ax[1].set_title('After alignment')
ax[1].set_aspect('equal')
ax[1].legend(markerscale=1)

## Save results

In [ ]:
df_aligned = pd.DataFrame({'aligned_x': aligned[:, 0], 'aligned_y': aligned[:, 1]})
results = pd.concat([df1, df_aligned], axis=1)
results.head()